To access the data from the notebook, replace the config.yaml file to point to the correct data path.

In [ ]:
import os
os.getcwd()

from pathlib import Path
from stgae.config.load_config import load_config
from stgae.data.preproccesing import get_columns
import pandas as pd

paths = load_config()['paths']    
data_path = Path(paths['raw_data'])

df = pd.read_csv(data_path / 'data.txt', names=get_columns(), sep=' ')

print(df.head())

In this dataset, an epoch is a discretization of time. The sensors were programmed to wake up, sense, and transmit data periodically (roughly every 31 seconds). However, because these are independent, low-power Mica2Dot motes, there are epochs with more than one observation per sensor. Therefore, I grouped observations by (sensor, epoch) pair. In particular, I kept the first available observation.

In [ ]:
# i first sort by datetime, to make sure that then first() means earliest
dt = pd.to_datetime(df['date'].astype(str) + ' ' + df['time'].astype(str),
                    errors='coerce')
df['datetime'] = dt
# drop rows that failed to parse
df = df.dropna(subset=['datetime']).reset_index(drop=True)

df = df.sort_values(by=['datetime'])
df = df.groupby(['epoch', 'moteid'], as_index=False).first()

Then, I decided that keeping different timestamps for different sensor inside the same epoch would add only noise and complexity to the model, as I effectively am modelling the problem as one observation per epoch (time series). Therefore, I aggregated datetime per epoch. In particular, for each epoch I calculated the median datetime, and set as datetime that median value.

In [ ]:
import numpy as np

df['datetime'] = df.groupby('epoch')['datetime'].transform(
    lambda x: pd.to_datetime(x.astype(np.int64).median())
)

I encoded dates and times using cyclical encoding. I encoded time of the day using a cylical encoding with period 24, and 

In [ ]:
import numpy as np

dt = pd.to_datetime(df['datetime'])

# split into date and time types
df['date'] = dt.dt.date
df['time'] = dt.dt.time

# hours
df['time_hours'] = dt.dt.hour + dt.dt.minute / 60 + dt.dt.second / 3600
df['time_sin'] = np.sin(2 * np.pi * df['time_hours'] / 24)
df['time_cos'] = np.cos(2 * np.pi * df['time_hours'] / 24)

#day of week
df['day_of_week'] = dt.dt.dayofweek
df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

#day of month
df['day_of_month'] = dt.dt.day
df['day_month_sin'] = np.sin(2 * np.pi * (df['day_of_month'] - 1) / 31)
df['day_month_cos'] = np.cos(2 * np.pi * (df['day_of_month'] - 1) / 31)

# drop extra columns
df = df.drop(columns=['time_hours', 'date', 'time', 'day_of_week',
                      'day_of_month', 'datetime'])

I dropped all the rows with missing data about the sensor id (few)

Now I count the average number of sensors per epoch. 

In [ ]:
# drop rows where moteid is NaN
df = df.dropna(subset=['moteid']).reset_index(drop=True) #526 observations
df.groupby('epoch').size().mean()

We have approximately 33.5 measurements per epoch. Given that the original data comes from 54 sensors... I study if there are particular sensors that have very little data.

In [ ]:
number_of_epochs = df['epoch'].nunique()
(df.groupby('moteid').size().sort_values() / number_of_epochs).iloc[:20]

I observed that there are some possibly mislabled sensors (65407, 6485, 33117, 56, 58, 57, 55) and some sensors with very little data (5, 18, 50, 8, 12). I discarded them entirely.

In [ ]:
discard = [65407, 6485, 33117, 56, 58, 57, 5, 18, 50, 8, 12, 55]
df = df[~df['moteid'].isin(discard)]
df['moteid'].unique()

I observed that feature light is missing in plenty observations.

In [ ]:
df.isna().sum() / len(df)

In [ ]:
import matplotlib.pyplot as plt

missing_counts = df.groupby('moteid')['light'].apply(lambda x: x.isna().sum()).sort_index()

plt.figure(figsize=(12,4))
missing_counts.plot(kind='bar')
plt.xlabel('Sensor (moteid)')
plt.ylabel('Number of missing "light" observations')
plt.title('Missing "light" observations per sensor')
plt.tight_layout()
plt.show()

Sensor 28 has too many observations where the feature light is missing. I discard it as well.

In [ ]:
df = df[df['moteid'] != 28]

df.isna().sum() / len(df)

Remaining missing data is little compared to the size of the dataset. I drop the rows with missing data. Then, I observed that there are epochs with more than one observation per sensors. This happens because epochs is in the end a (given by the dataset) discretization of time. Therefore, in order to treat this as a time series, I aggregated (averaged) observed features per epoch.

In [ ]:
df = df.dropna().reset_index(drop=True)

Finally I reindexed epochs and sensors ids.

In [ ]:
#rename moteid
moteid_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(df['moteid'].unique()))}
df['moteid'] = df['moteid'].map(moteid_mapping)

first_epoch = df['epoch'].min()
df['epoch'] = df['epoch'] - first_epoch

assert df['epoch'].max() == df['epoch'].nunique()-1 #all epochs are present
print(len(df['epoch'].unique()))

In [ ]:
df.describe()

I could quickly see that the data contains errors: for example, the maximum temperature is 385 degrees Celsius. To fix this errors that would cause my model to learn from them and thus make it predict garbage, I decided to set a range of physically reasonable values for all features, and drop rows with observed values outside that range. I defined the following intervals:
- Voltage: [2, 3.2]. This comes from the dataset definition.
- Temperature: [8, 38]. Since it is an office environemnt, temperatures outside this range are impossible.
- Humidity: [0, 100]. The only valid theoretical range.
- Light: [0, 2000]. 0 is darkness. 2000 is very bright direct light. The sensors often glitch to extremely high negative or positive integers.

In [ ]:
from stgae.data.preproccesing import filter_data_errors

df = filter_data_errors(df)

In [ ]:
# drop rows where moteid is NaN
df.groupby('epoch').size().mean()

To model the data as a graph, I built the adjacency matrix for the remaining sensors.
I constructed the graph following a kNN approach. I defined the adjacency matrix such that an edge (v, w) was included in it if w was one of the k nearest neighbors of v or viceversa. I followed this “or” approach to ensure the matrix was symmetric, and therefore the resulting graph undirected, which was more natural for a distance-based graph and because the original STGCN model was built for an undirected graph.
Then, I added weights to the matrix, calculated as the inverse of the distance between the connected nodes. Additionally I added self-loops: they ensure that the neighboring nodes retain their true values to act as  anchors for the imputation, rather than becoming "blurry" averages. Furthermore, they preserve the target node's valid historical data (at previous timesteps) from being smoothed away by the spatial module. I set the self-loop weight to be the equal to the maximum weight of its neighbors.
Lastly,  and also following the STGCN method, I normalized the weights of the matrix per node, that is, I normalized the weights for each node so that the sum of all its edges summed to 1.

In [ ]:
from stgae.data.preproccesing import calculate_distances, calculate_adjacency_matrix

coordinates = pd.read_csv(paths['data_root'] / 'sensor_coordinates.txt', sep=' ')

dist_matrix = calculate_distances(coordinates) #numpy array

k=3 #for k-nearest neighbors adj matrix
exclude_sensors = [28, 5, 18, 50, 8, 12]

A = calculate_adjacency_matrix(dist_matrix, k=k, exclude=exclude_sensors) #assumes renaming of moteid, according to excluded sensors

assert A.shape[0] == len(df['moteid'].unique())

Now, I built the tensors to then create the Dataset. <br> To build the input tensors, I separated columns in target columns and time columns.
Target columns are the ones that I am interested to predict, namely temperature, humidity, voltage and light, whereas time related columns (encoded time features) will be fed to the model, but will not be predicted.
Additionally, I created a “mask” tensor, that for every epoch and every sensor indicates if there is data for that sensor at that epoch.  Then I divided the tensors into training, validation and evaluation sets, keeping temporal order.
Finally, I normalized the data defining a custom Masked Standard Scaler. I calculated mean and variances in the training set for each feature, excluding the missing values, and transformed  all sets with the scaler fitted in the training set. This way,  masking a given feature value becomes equivalent to setting it to zero, because it is the standardized mean value.

In [ ]:
from stgae.data.preproccesing import build_tensors

#shapes:
#X: (time_steps, num_sensors, num_features), where X[t, n] = features of sensor n at epoch t
# M: (time_steps, num_sensors), where M[t, n] = 1 if sensor exists at epoch t, else 0
# T_enc: (time_steps, num_time_features)
X, M, T_enc = build_tensors(df, epochs=df['epoch'].unique(), sensors=df['moteid'].unique(), feature_cols=['temperature', 'humidity', 'light', 'voltage'], time_cols=['time_sin', 'time_cos', 'day_sin', 'day_cos', 'day_month_sin', 'day_month_cos'])

I defined a custom dataset (from module Dataset)
With the custom dataset class I implemented the sliding window and the masking logic. Each get item method returns:
x_past_masked and x_future_masked: past and future windows of shape (W+1, N, F_target), with a set of sensors features at time t masked (set to zero, following the previous standard normalization). In the training set masking is done stochastically, while in the test set masking is done deterministically.
time_past and time_future: past and future windows time features of shape (W+1, F_time), not masked.
target: not masked state at time t, of shape (N, F_target)
time_target: time features at time t, of shape (F_time)
loss_mask: binary array of shape N, indicating which sensors were masked at time t


In [ ]:
from stgae.data.dataset import STBGNNDataset
from stgae.data.preproccesing import MaskedStandardScaler

train_split = 0.75
val_split = 0.1
T = X.shape[0]
train_size = int(T * train_split)
val_size = int(T * val_split)

#divide into train and val, keeping temporal order, to avoid leakage and learn temporal dependencies
X_train = X[:train_size]
X_val = X[train_size:train_size + val_size]
M_train = M[:train_size]
M_val = M[train_size:train_size + val_size]
T_enc_train = T_enc[:train_size] 
T_enc_val = T_enc[train_size:train_size+val_size]

#test dataset will be held out only for final evaluation, not for model selection
X_test = X[train_size + val_size:]
M_test = M[train_size + val_size:]
T_enc_test = T_enc[train_size + val_size :]

#i fit the scalar on the training data only, exxcludin the structurally missing data
scaler = MaskedStandardScaler()
scaler.fit(X_train, M_train)

#i scale the tensors remasking missing values back to zero
train_norm = scaler.transform(X_train, M_train, remask_zeros=True)
val_norm = scaler.transform(X_val, M_val, remask_zeros=True)
test_norm = scaler.transform(X_test, M_test, remask_zeros=True)

train_dataset = STBGNNDataset(train_norm, M_train, T_enc, window_size=4, mask_ratio=0.5, split="train")
val_dataset = STBGNNDataset(val_norm, M_val, T_enc_val, window_size=4, mask_ratio=0.5, split="test")
test_dataset = STBGNNDataset(test_norm, M_test, T_enc_test, window_size=4, mask_ratio=0.5, split="test")

print('train dataset length:', len(train_dataset))
print('val dataset length:', len(val_dataset))
print('test dataset length:', len(test_dataset))

Note that masking in the training dataset is stochastic, while masking in the test dataset is deterministic. This is implemented using a local_rng for the test dataset (see code)

Training the model

In [ ]:
import torch
from torch.utils.data import DataLoader
from stgae.model.bistgcn_opt import BiSTGCN
from stgae.model.train import train_step
from stgae.config.load_config import load_config
from stgae.model.evaluate import evaluate
from stgae.utils.utils import get_device

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 0.001
HIDDEN_DIM = 32
DEVICE = get_device()
SAVE_DIR = load_config()["paths"]["checkpoints"] #dir to store models

os.makedirs(SAVE_DIR, exist_ok=True)

# dataloader
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True) #this does not shuffle timesteps inside samples, but samples inside the batches and improves SGD 
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = BiSTGCN(target_dim=train_dataset.F, time_dim= train_dataset.F_time, hidden_dim=HIDDEN_DIM, adj=A).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_loss = float('inf')
train_losses = []
val_mses = []

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    for batch_idx, batch in enumerate(train_dataloader):
        loss = train_step(batch, model, optimizer, DEVICE)
        train_loss += loss

        if batch_idx % 500 == 200:
            print(f"   Batch {batch_idx+1}/{len(train_dataloader)} - Train Loss: {loss:.6f}")
    avg_train_loss = train_loss / len(train_dataloader)

    val_mse, val_mae, val_rmse = evaluate(val_dataloader, model, DEVICE)

    train_losses.append(avg_train_loss)
    val_mses.append(val_mse)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"   Train Loss: {avg_train_loss:.6f}")
    print(f"   Val MSE:    {val_mse:.6f}")
    print(f"   Val MAE:    {val_mae:.6f}")
    print(f"   Val RMSE:   {val_rmse:.6f}")

    if val_mse < best_loss:
        best_loss = val_mse
        best_path = os.path.join(SAVE_DIR, "bistgcn_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"   -> New best model saved with MSE {val_mse} to {best_path}")

    # save for resuming if crash
    checkpoint_path = os.path.join(SAVE_DIR, "bistgcn_last.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train_loss,
        'loss': val_mse,
    }, checkpoint_path)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, label='Train MSE', marker='o')
plt.plot(epochs, val_mses, label='Validation MSE', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Validation Loss over Epochs')
plt.legend()
plt.grid(True)
plt.show()

Evaluating the model.
Now I evaluated the trained model in the test dataset.

In [ ]:
import torch
from torch.utils.data import DataLoader
from stgae.model.bistgcn_opt import BiSTGCN
from stgae.model.train import train_step
from stgae.config.load_config import load_config
from stgae.model.evaluate import evaluate
from stgae.utils.utils import get_device

import torch
#load model
BATCH_SIZE = 32
CHECKPOINTS = load_config()["paths"]["checkpoints"]
HIDDEN_DIM = 32
DEVICE = get_device()


model_path = os.path.join(CHECKPOINTS, "bistgcn_best.pth")
model = BiSTGCN(target_dim=train_dataset.F, time_dim= train_dataset.F_time, hidden_dim=HIDDEN_DIM, adj=A).to(DEVICE)
checkpoint = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(checkpoint)
model.eval()

test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

eval_mse, eval_mae, eval_rmse = evaluate(test_dataloader, model, DEVICE)

print(f'MSE = {eval_mse} | MAE =  {eval_mae} | RMSE = {eval_rmse}')

Then, to analyze the model, I visualized the reconstruction. I defined a function to visualize the reconstruction for a chosen sensor in the evaluation dataset, in the masked data points.

In [ ]:
from stgae.model.visualize import visualize_sensor_reconstruction

#labels
feature_labels = ["Temperature (°C)", "Humidity (%)", "Light (Lux)", "Voltage (V)"]

sensor_to_plot = 10

# test_loader uast be shuffle=False so the plot is  continuous 
visualize_sensor_reconstruction(
    dataloader=test_dataloader,  
    model=model, 
    scaler=scaler, 
    sensor_idx=sensor_to_plot, 
    feature_names=feature_labels, 
    device=DEVICE,
    num_steps=9508 
)